# Notebook 07 — Augmented Retraining

## Motivation

NB03 training had two known gaps identified through NB04–NB06C evaluation:

1. **Sparse negatives (964)** — caused high FP rate at default threshold on real traffic
2. **Evasion variants missing from positives** — obfuscated payloads score 0.79–0.96, below T_low=0.99

## Fixes

- **Fix 1**: Expand negatives with 1,523 real benign query values extracted from `access-generated.log`
- **Fix 2**: Augment positives with known evasion techniques (tab encoding, mixed-case, CHR(), SLEEP, etc.)
- **Fix 3**: Re-tune thresholds on augmented data

## Evaluation

Before/after comparison on evasion payloads from live pipeline testing.

## 1. Imports & Setup

In [3]:
import pandas as pd
import numpy as np
import joblib, os, re, time, json, urllib.parse, warnings, random
warnings.filterwarnings('ignore')

from collections import Counter
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, precision_recall_curve
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

for d in ['results/models', 'results/figures', 'results/metrics']:
    os.makedirs(d, exist_ok=True)

random.seed(42)
np.random.seed(42)
print('Setup complete.')

Setup complete.


## 2. Core Functions (identical to NB03)

In [5]:
SYMBOLS = ["'",'"',";","--","#","/*","*/","*","+","|","(",")",">","<","\\","/","="]

def extract_symbol_frequencies(text):
    c = Counter()
    for sym in SYMBOLS:
        c[sym] = str(text).count(sym)
    return c

def build_symbol_matrix(queries):
    rows = [extract_symbol_frequencies(q) for q in queries]
    return csr_matrix(pd.DataFrame(rows).fillna(0).values)

def extract_query_values(url):
    parsed = urllib.parse.urlparse(url)
    params = urllib.parse.parse_qs(parsed.query, keep_blank_values=False)
    values = [
        urllib.parse.unquote(v).strip()
        for vlist in params.values()
        for v in vlist
        if urllib.parse.unquote(v).strip()
    ]
    return ' '.join(values) if values else None

print('Core functions defined.')

Core functions defined.


## 3. Positive Examples

### Part A — NB03 baseline positives (same as before)
### Part B — Evasion augmentation (new in NB07)

In [7]:
TEMPLATES = [
    '/product?id={}',
    '/search?q={}',
    '/user?name={}',
    '/item?id={}',
    '/api/v1?input={}',
    '/filter?category={}',
    '/login?user={}',
    '/page?content={}',
    '/data?value={}',
    '/query?term={}',
]

# ── Part A: NB03 baseline positives ──────────────────────────────
df_clean     = pd.read_csv('../datasets/Cleaned_SQL_Dataset.csv')
sqli_queries = df_clean[df_clean['Label'] == 1]['Query'].tolist()

positives = []
for i, q in enumerate(sqli_queries):
    tmpl = TEMPLATES[i % len(TEMPLATES)]
    url  = tmpl.format(urllib.parse.quote(str(q), safe="'\"=;-/*+()|<>\\#@ "))
    val  = extract_query_values(url)
    if val:
        positives.append(val)

print(f'Part A — Baseline positives : {len(positives):,}')

Part A — Baseline positives : 11,383


In [8]:
# ── Part B: Evasion augmentation ─────────────────────────────────
# These are the specific techniques that scored 0.79–0.96 in live pipeline testing
# and were missed by the NB03 model.

evasion_payloads = [

    # 1. Mixed-case + comment insertion
    "1'/**/uNiOn/**/sElEcT/**/1,2,3--",
    "1'/**/UnIoN/**/SeLeCt/**/username,password/**/FrOm/**/users--",
    "'/**/oR/**/1/**/=/**/1--",
    "admin'/**/aNd/**/1=1--",
    "1/**/UNION/**/ALL/**/SELECT/**/NULL,NULL,NULL--",

    # 2. Tab / whitespace encoding
    "1%09OR%091=1",
    "1%09UNION%09SELECT%091,2,3--",
    "'%09OR%09'1'='1",
    "1%0AOR%0A1=1--",
    "1%0D%0AOR%0D%0A1=1--",

    # 3. Scientific notation tautologies
    "1 OR 1e0=1e0--",
    "1 AND 1e0=1e0--",
    "' OR 1e0=1e0--",
    "1 OR 2e0>1e0--",

    # 4. Version-conditional MySQL comments
    "1/*!50000OR*/1=1",
    "1/*!50000UNION*//*!50000SELECT*/1,2,3--",
    "'/*!50000OR*/'1'='1",
    "1/*!32302OR*/1=1--",

    # 5. Time-based blind — SLEEP
    "1 AND SLEEP(5)--",
    "1' AND SLEEP(5)--",
    "1 OR SLEEP(5)--",
    "1; SELECT SLEEP(5)--",
    "' OR SLEEP(5)='0",
    "1 AND (SELECT * FROM (SELECT(SLEEP(5)))a)--",

    # 6. Time-based blind — pg_sleep (PostgreSQL)
    "1; SELECT pg_sleep(5)--",
    "'; SELECT pg_sleep(5)--",
    "1 AND 1=(SELECT 1 FROM pg_sleep(5))--",

    # 7. Time-based blind — WAITFOR (MS-SQL)
    "1'; WAITFOR DELAY '0:0:5'--",
    "1; WAITFOR DELAY '0:0:5'--",
    "'; WAITFOR DELAY '0:0:5'--",

    # 8. Stacked queries
    "1; DROP TABLE users--",
    "1; SELECT * FROM users--",
    "1'; INSERT INTO users VALUES('hacker','hacker')--",
    "1; UPDATE users SET password='hacked' WHERE 1=1--",
    "1; DELETE FROM logs--",

    # 9. CHR() / CHAR() encoding
    "admin' OR user=CHAR(97,100,109,105,110)--",
    "' OR CHAR(49)=CHAR(49)--",
    "1 UNION SELECT CHAR(117,115,101,114),CHAR(112,97,115,115)--",
    "' OR CHR(49)=CHR(49)--",

    # 10. Hex encoding
    "' OR name=0x61646d696e--",
    "' OR 1=0x31--",
    "1 UNION SELECT 0x61646d696e,0x70617373--",

    # 11. Nested subqueries
    "(SELECT(SELECT(SELECT(1))))",
    "1 AND (SELECT 1 FROM (SELECT COUNT(*),CONCAT(version(),FLOOR(RAND(0)*2))x FROM users GROUP BY x)a)",
    "(SELECT 1 FROM (SELECT SLEEP(5))a)",
    "1 AND (SELECT 1 FROM users WHERE username='admin' AND SLEEP(3))",

    # 12. Double URL encoding
    "1%2527%2520OR%25201%253D1",
    "'%2520OR%25201=1--",

]

# Wrap evasion payloads in URL templates and extract query values
evasion_positives = []
for i, payload in enumerate(evasion_payloads):
    tmpl = TEMPLATES[i % len(TEMPLATES)]
    url  = tmpl.format(urllib.parse.quote(str(payload), safe="'\"=;-/*+()|<>\\#@% "))
    val  = extract_query_values(url)
    if val:
        evasion_positives.append(val)

# Also add raw decoded forms (some payloads after URL decoding)
raw_evasion = [
    "1 OR 1=1",                          # decoded tab encoding
    "1 UNION SELECT 1,2,3--",            # decoded tab-encoded UNION
    "1 AND SLEEP(5)",
    "1 OR SLEEP(5)",
    "1 SELECT pg_sleep(5)",
    "1 WAITFOR DELAY 0:0:5",
    "1 DROP TABLE users",
    "1 INSERT INTO users VALUES hacker",
]
evasion_positives += raw_evasion

print(f'Part B — Evasion positives  : {len(evasion_positives):,}')
print()
print('Sample evasion query values (what the model sees):')
for v in evasion_positives[:8]:
    print(f'  {repr(v[:80])}')

Part B — Evasion positives  : 56

Sample evasion query values (what the model sees):
  "1'/**/uNiOn/**/sElEcT/**/1,2,3--"
  "1'/**/UnIoN/**/SeLeCt/**/username,password/**/FrOm/**/users--"
  "'/**/oR/**/1/**/=/**/1--"
  "admin'/**/aNd/**/1=1--"
  '1/**/UNION/**/ALL/**/SELECT/**/NULL,NULL,NULL--'
  '1\tOR\t1=1'
  '1\tUNION\tSELECT\t1,2,3--'
  "'\tOR\t'1'='1"


In [9]:
# ── Combine all positives ─────────────────────────────────────────
all_positives = list(dict.fromkeys(positives + evasion_positives))
print(f'Total positives (deduped) : {len(all_positives):,}')
print(f'  Baseline                : {len(positives):,}')
print(f'  Evasion augmentation    : {len(evasion_positives):,}')

Total positives (deduped) : 11,215
  Baseline                : 11,383
  Evasion augmentation    : 56


## 4. Negative Examples

### Part A — Real benign query values from access-generated.log (new in NB07)
### Part B — NB03 synthetic negatives (retained for coverage)

In [11]:
# NOTE: The raw extracted values contained 44 attack payloads
# (XSS, path traversal, SQLi variants) from the generated log.
# These were removed using pattern-based filtering before use as negatives.
# See: real_benign_qvalues_clean.json (1,479 entries, down from 1,523)
# ── Part A: Real benign query values ─────────────────────────────
# Extracted from access-generated.log — 1,523 unique values
# All confirmed benign (grep-filtered, no SQL keywords)
# Real traffic scores confirmed < 0.30 on NB03 model in live pipeline testing

with open('results/benign_pool/07_real_benign_qvalues_clean.json', 'r') as f:
    real_negatives = json.load(f)

print(f'Part A — Real benign negatives   : {len(real_negatives):,}')
print('Sample:')
for v in real_negatives[:8]:
    print(f'  {repr(v)}')

Part A — Real benign negatives   : 1,479
Sample:
  '1'
  '1 1'
  '1 10'
  '1 123'
  '1 2'
  '1 20'
  '1 25'
  '1 3'


In [12]:
# ── Part B: NB03 synthetic negatives (retained) ───────────────────
base_ids     = [str(i) for i in range(1, 200)]
page_nums    = [str(i) for i in range(1, 51)]
sort_vals    = ['desc','asc','price desc','price asc','created desc','created asc',
                'newest','oldest','popular','relevance','default','rating desc']
filter_vals  = ['active','all','none','enabled','true','false','verified','new','used']
categories   = ['electronics','furniture','cars','jobs','services','real-estate',
                'mobiles','laptops','clothes','sports','books','cameras']
locations    = ['cairo','dubai','riyadh','amman','baghdad','tunis','casablanca',
                'kuwait','doha','muscat','abu-dhabi','beirut','algiers','damascus']
search_terms = ['laptop','mobile','car','apartment','samsung','iphone','toyota',
                'sofa','chair','camera','watch','shoes','dress','bicycle','tablet']

synthetics = []
synthetics += base_ids
synthetics += ['page ' + p for p in page_nums]
synthetics += sort_vals + filter_vals + categories + locations + search_terms
synthetics += [i + ' ' + s for i in base_ids[:50] for s in sort_vals[:3]]
synthetics += [t + ' ' + l for t in search_terms for l in locations[:5]]
synthetics += [c + ' ' + s for c in categories for s in sort_vals[:4]]
synthetics += [c + ' ' + s + ' ' + p
               for c in categories
               for s in sort_vals[:3]
               for p in page_nums[:5]]
synthetics += [str(i) for i in range(10000, 10200)]

SQL_KW = ["' ","select ","union ","insert ","drop ","delete "," or "," and "]
synthetics = list(dict.fromkeys(synthetics))
synthetics = [s for s in synthetics if not any(k in s.lower() for k in SQL_KW)]

print(f'Part B — Synthetic negatives     : {len(synthetics):,}')

# ── Combine all negatives ──────────────────────────────────────────
all_negatives = list(dict.fromkeys(real_negatives + synthetics))
print(f'Total negatives (deduped)        : {len(all_negatives):,}')
print(f'  NB03 had                       : 964')
print(f'  NB07 has                       : {len(all_negatives):,}  (+{len(all_negatives)-964:,})')

Part B — Synthetic negatives     : 964
Total negatives (deduped)        : 2,426
  NB03 had                       : 964
  NB07 has                       : 2,426  (+1,462)


## 5. Combine & Check Balance

In [14]:
all_queries = all_positives + all_negatives
all_labels  = [1] * len(all_positives) + [0] * len(all_negatives)

pos = sum(all_labels)
neg = len(all_labels) - pos
print(f'Positives : {pos:,}  (NB03 baseline + evasion augmentation)')
print(f'Negatives : {neg:,}  (real traffic + synthetic)')
print(f'Total     : {len(all_queries):,}')
print(f'Ratio     : 1:{neg/pos:.2f}  (NB03 was 1:0.08)')
print()
print('Note: improved ratio reduces positive-class bias at default threshold.')

Positives : 11,215  (NB03 baseline + evasion augmentation)
Negatives : 2,426  (real traffic + synthetic)
Total     : 13,641
Ratio     : 1:0.22  (NB03 was 1:0.08)

Note: improved ratio reduces positive-class bias at default threshold.


## 6. Split First → Vectorize (Train-Only Fit)

In [16]:
q_train, q_test, y_train, y_test = train_test_split(
    all_queries, all_labels, test_size=0.2, random_state=42, stratify=all_labels)

vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 3), min_df=2)
vectorizer.fit(q_train)

X_train = hstack([vectorizer.transform(q_train), build_symbol_matrix(q_train)])
X_test  = hstack([vectorizer.transform(q_test),  build_symbol_matrix(q_test)])

print(f'Train size : {X_train.shape[0]:,}')
print(f'Test size  : {X_test.shape[0]:,}')
print(f'Features   : {X_train.shape[1]:,}  (char n-grams + 17 symbols)')
print(f'  NB03 had : 15,202 features')

Train size : 10,912
Test size  : 2,729
Features   : 8,824  (char n-grams + 17 symbols)
  NB03 had : 15,202 features


## 7. Train Random Forest

In [18]:
print('Training Random Forest...')
t0 = time.perf_counter()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
train_ms = (time.perf_counter() - t0) * 1000
print(f'Training time: {train_ms/1000:.1f}s')

y_prob = rf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

f1    = f1_score(y_test, y_pred)
prec  = precision_score(y_test, y_pred)
rec   = recall_score(y_test, y_pred)
prauc = average_precision_score(y_test, y_prob)

# E2E latency — single request
sample = [q_test[0]]
t0 = time.perf_counter()
Xs = hstack([vectorizer.transform(sample), build_symbol_matrix(sample)])
rf.predict_proba(Xs)
e2e_ms = (time.perf_counter() - t0) * 1000

print(f'\nRF Test Set Results:')
print(f'  F1        : {f1:.4f}  (NB03: 0.9982)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  PR-AUC    : {prauc:.4f}  (NB03: 1.0000)')
print(f'  E2E       : {e2e_ms:.1f}ms  (NB03: 65.8ms)')

Training Random Forest...
Training time: 0.9s

RF Test Set Results:
  F1        : 0.9982  (NB03: 0.9982)
  Precision : 0.9987
  Recall    : 0.9978
  PR-AUC    : 1.0000  (NB03: 1.0000)
  E2E       : 47.0ms  (NB03: 65.8ms)


## 8. Threshold Tuning

In [20]:
p_c, r_c, thr = precision_recall_curve(y_test, y_prob)
pairs = sorted(zip(thr, p_c[:-1], r_c[:-1]), key=lambda x: -x[0])

t_high = p_high = r_high = None
for t, p, r in pairs:
    if p >= 0.995:
        t_high, p_high, r_high = round(float(t),3), round(float(p),4), round(float(r),4)
        break

t_low = p_low = r_low = None
for t, p, r in pairs:
    if r >= 0.95:
        t_low, p_low, r_low = round(float(t),3), round(float(p),4), round(float(r),4)
        break

print('=== THRESHOLD TUNING (RF) ===')
print(f'T_high : {t_high}  prec={p_high}  rec={r_high}  (NB03: T_high=1.0)')
print(f'T_low  : {t_low}   prec={p_low}   rec={r_low}   (NB03: T_low=0.99)')
print(f'T_high > T_low : {t_high} > {t_low}  {"✅" if t_high > t_low else "❌"}')

th_dict = {
    't_high': t_high, 'p_high': p_high, 'r_high': r_high,
    't_low':  t_low,  'p_low':  p_low,  'r_low':  r_low,
}
with open('results/models/07_thresholds.json', 'w') as f:
    json.dump(th_dict, f, indent=2)
print('Saved: results/models/07_thresholds.json')

=== THRESHOLD TUNING (RF) ===
T_high : 1.0  prec=1.0  rec=0.9184  (NB03: T_high=1.0)
T_low  : 0.99   prec=1.0   rec=0.9626   (NB03: T_low=0.99)
T_high > T_low : 1.0 > 0.99  ✅
Saved: results/models/07_thresholds.json


## 9. Evasion Payload Before/After Comparison

In [22]:
# Load NB03 model for before scores
rf_nb03 = joblib.load('results/models/03_random_forest_model.pkl')
vec_nb03 = joblib.load('results/models/03_vectorizer.pkl')

# These are the payloads that scored 0.79–0.96 in live pipeline testing
evasion_test = [
    ("Tab-encoded OR",           "1 OR 1=1"),
    ("Scientific notation",      "1 OR 1e0=1e0--"),
    ("Version comment",          "1/*!50000OR*/1=1"),
    ("Nested subquery",          "(SELECT(SELECT(SELECT(1))))"),
    ("Mixed-case UNION",         "1'/**/uNiOn/**/sElEcT/**/1,2,3--"),
    ("SLEEP time-based",         "1 AND SLEEP(5)--"),
    ("pg_sleep",                 "1; SELECT pg_sleep(5)--"),
    ("WAITFOR DELAY",            "1'; WAITFOR DELAY '0:0:5'--"),
    ("Stacked DROP TABLE",       "1; DROP TABLE users;--"),
    ("CHAR() encoding",          "admin' OR user=CHAR(97,100,109,105,110)--"),
    ("Hex encoding",             "' OR name=0x61646d696e--"),
    ("Double URL encoding",      "1%2527%2520OR%25201%253D1"),
]

def score_payload(payload, model, vectorizer):
    X = hstack([vectorizer.transform([payload]), build_symbol_matrix([payload])])
    return round(float(model.predict_proba(X)[0][1]), 4)

print(f'{"Technique":<25} {"Payload":<45} {"NB03":>6} {"NB07":>6} {"Δ":>6} {"NB03 Tier":<12} {"NB07 Tier"}')
print('-' * 120)

nb03_t_low = 0.99
nb03_t_high = 1.0

def tier(score, t_low, t_high):
    if score >= t_high: return 'ATTACK'
    if score >= t_low:  return 'SUSPICIOUS'
    return 'BENIGN'

results = []
for name, payload in evasion_test:
    s03 = score_payload(payload, rf_nb03, vec_nb03)
    s07 = score_payload(payload, rf,      vectorizer)
    delta = s07 - s03
    t03 = tier(s03, nb03_t_low, nb03_t_high)
    t07 = tier(s07, t_low, t_high)
    improved = '✅' if t07 != 'BENIGN' and t03 == 'BENIGN' else ('➡' if t07 == t03 else '⚠')
    print(f'{name:<25} {payload[:43]:<45} {s03:>6.3f} {s07:>6.3f} {delta:>+6.3f} {t03:<12} {t07} {improved}')
    results.append({'technique': name, 'nb03_score': s03, 'nb07_score': s07, 'delta': delta,
                    'nb03_tier': t03, 'nb07_tier': t07})

df_results = pd.DataFrame(results)
improved_count = len(df_results[df_results['nb07_tier'] != 'BENIGN'])
print(f'\nPayloads now detected (non-BENIGN): {improved_count}/{len(evasion_test)}')
df_results.to_csv('results/metrics/07_evasion_comparison.csv', index=False)

Technique                 Payload                                         NB03   NB07      Δ NB03 Tier    NB07 Tier
------------------------------------------------------------------------------------------------------------------------
Tab-encoded OR            1 OR 1=1                                       0.923  0.860 -0.063 BENIGN       BENIGN ➡
Scientific notation       1 OR 1e0=1e0--                                 0.960  0.960 +0.000 BENIGN       BENIGN ➡
Version comment           1/*!50000OR*/1=1                               0.790  0.990 +0.200 BENIGN       SUSPICIOUS ✅
Nested subquery           (SELECT(SELECT(SELECT(1))))                    0.924  0.840 -0.084 BENIGN       BENIGN ➡
Mixed-case UNION          1'/**/uNiOn/**/sElEcT/**/1,2,3--               0.910  1.000 +0.090 BENIGN       ATTACK ✅
SLEEP time-based          1 AND SLEEP(5)--                               0.980  1.000 +0.020 BENIGN       ATTACK ✅
pg_sleep                  1; SELECT pg_sleep(5)--                    

## 10. 5-Fold Cross-Validation

In [24]:
print('Running 5-fold CV...')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
qa, la = np.array(all_queries), np.array(all_labels)

cv_f1s, cv_praucs = [], []
for fold, (tr_idx, val_idx) in enumerate(skf.split(qa, la), 1):
    q_tr, q_val = qa[tr_idx].tolist(), qa[val_idx].tolist()
    y_tr, y_val = la[tr_idx],          la[val_idx]

    vec_cv = CountVectorizer(analyzer='char', ngram_range=(1, 3), min_df=2)
    vec_cv.fit(q_tr)
    X_tr  = hstack([vec_cv.transform(q_tr),  build_symbol_matrix(q_tr)])
    X_val = hstack([vec_cv.transform(q_val), build_symbol_matrix(q_val)])

    rf_cv = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_cv.fit(X_tr, y_tr)

    y_vp = rf_cv.predict_proba(X_val)[:, 1]
    y_vl = (y_vp >= 0.5).astype(int)

    cv_f1s.append(f1_score(y_val, y_vl))
    cv_praucs.append(average_precision_score(y_val, y_vp))
    print(f'  Fold {fold}: F1={cv_f1s[-1]:.4f}  PR-AUC={cv_praucs[-1]:.4f}')

print(f'\nCV F1    : {np.mean(cv_f1s):.4f} ± {np.std(cv_f1s):.4f}  (NB03: 0.9982 ± 0.0006)')
print(f'CV PR-AUC: {np.mean(cv_praucs):.4f} ± {np.std(cv_praucs):.4f}  (NB03: 1.0000)')

Running 5-fold CV...
  Fold 1: F1=0.9975  PR-AUC=1.0000
  Fold 2: F1=0.9980  PR-AUC=1.0000
  Fold 3: F1=0.9978  PR-AUC=1.0000
  Fold 4: F1=0.9982  PR-AUC=1.0000
  Fold 5: F1=0.9980  PR-AUC=1.0000

CV F1    : 0.9979 ± 0.0002  (NB03: 0.9982 ± 0.0006)
CV PR-AUC: 1.0000 ± 0.0000  (NB03: 1.0000)


## 11. Save Model & Vectorizer

In [26]:
joblib.dump(rf,         'results/models/07_rf_model.pkl')
joblib.dump(vectorizer, 'results/models/07_vectorizer.pkl')
print('Saved: results/models/07_rf_model.pkl')
print('Saved: results/models/07_vectorizer.pkl')
print('Saved: results/models/07_thresholds.json')

Saved: results/models/07_rf_model.pkl
Saved: results/models/07_vectorizer.pkl
Saved: results/models/07_thresholds.json


## 12. Summary

In [28]:
th = json.load(open('results/models/07_thresholds.json'))
print('=' * 65)
print('NOTEBOOK 07 — COMPLETE (Augmented Retraining)')
print('=' * 65)
print()
print('DATASET COMPARISON:')
print(f'  NB03 positives : 11,383   NB07: {len(all_positives):,}')
print(f'  NB03 negatives :    964   NB07: {len(all_negatives):,}')
print(f'  NB03 total     : 12,347   NB07: {len(all_queries):,}')
print(f'  NB03 ratio     : 1:0.08   NB07: 1:{len(all_negatives)/len(all_positives):.2f}')
print()
print('THRESHOLD COMPARISON:')
print(f'  NB03 T_high=1.0   T_low=0.99')
print(f'  NB07 T_high={th["t_high"]}  T_low={th["t_low"]}')
print()
print('EVASION DETECTION:')
improved = df_results[df_results['nb07_tier'] != 'BENIGN']
print(f'  NB03: 0/{len(evasion_test)} evasion payloads detected')
print(f'  NB07: {len(improved)}/{len(evasion_test)} evasion payloads detected')
print()
print('NEXT: Update pipeline to use NB07 model and re-evaluate on real logs')

NOTEBOOK 07 — COMPLETE (Augmented Retraining)

DATASET COMPARISON:
  NB03 positives : 11,383   NB07: 11,215
  NB03 negatives :    964   NB07: 2,426
  NB03 total     : 12,347   NB07: 13,641
  NB03 ratio     : 1:0.08   NB07: 1:0.22

THRESHOLD COMPARISON:
  NB03 T_high=1.0   T_low=0.99
  NB07 T_high=1.0  T_low=0.99

EVASION DETECTION:
  NB03: 0/12 evasion payloads detected
  NB07: 5/12 evasion payloads detected

NEXT: Update pipeline to use NB07 model and re-evaluate on real logs


In [ ]:
import json
import pandas as pd
import os
import urllib.parse

os.makedirs('./results/training_data', exist_ok=True)

# ── Positives — SQLi payloads from Cleaned_SQL_Dataset ───────────
df = pd.read_csv('../datasets/Cleaned_SQL_Dataset.csv')
positives = df[df['Label'] == 1]['Query'].dropna().tolist()
positives = [str(p).strip() for p in positives if str(p).strip()]

# ── Negatives — clean real benign query values from NB07 ─────────
with open('./results/benign_pool/07_real_benign_qvalues_clean.json', 'r') as f:
    real_benign = json.load(f)

# Also add synthetic negatives from NB03
synthetic = [
    str(i) for i in range(1, 200)          # numeric IDs
] + [
    str(i) for i in range(10000, 10200)    # larger IDs
] + [
    'asc', 'desc', 'price asc', 'price desc',
    'name asc', 'name desc', 'date asc', 'date desc',
    'newest', 'oldest', 'popular', 'featured',
    'electronics', 'furniture', 'clothing', 'books',
    'cairo', 'alexandria', 'london', 'paris',
    'search', 'filter', 'sort', 'page',
]

negatives = list(set(real_benign + synthetic))
negatives = [str(n).strip() for n in negatives if str(n).strip()]

# ── Save ─────────────────────────────────────────────────────────
with open('./results/training_data/07_positives.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(positives))

with open('./results/training_data/07_negatives.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(negatives))

print(f'Positives saved : {len(positives):,}')
print(f'Negatives saved : {len(negatives):,}')
print('Files saved to notebooks/results/training_data/')

Positives saved : 11,386
Negatives saved : 1,887
Files saved to notebooks/results/training_data/


: 